# OpenACC in Flang's MLIR pipeline

This notebook records the boundary on current LLVM main honestly: Flang parses OpenACC and lowers it to the MLIR OpenACC dialect, but basic `parallel`, `serial`, and `data` regions still abort in the later OpenACC-to-LLVM path. Therefore this is an inspectable frontend/MLIR demonstration, not an executable OpenACC benchmark yet.

## Fortran input

```fortran
program test
  integer :: x
  x = 0
  !$acc parallel
  x = 1
  !$acc end parallel
  print *, x
end program
```

## HLFIR emitted by current Flang

Generated with `flang -fc1 -fopenacc -emit-hlfir`:

```mlir
func.func @_QQmain() {
  %x = fir.alloca i32 <{bindc_name = "x"}>
  %decl:2 = hlfir.declare %x : (!fir.ref<i32>) -> (!fir.ref<i32>, !fir.ref<i32>)
  %zero = arith.constant 0 : i32
  hlfir.assign %zero to %decl#0 : i32, !fir.ref<i32>
  acc.parallel {
    %one = arith.constant 1 : i32
    hlfir.assign %one to %decl#0 : i32, !fir.ref<i32>
    acc.yield
  }
  return
}
```

## Next milestone

Once OpenACC-to-LLVM handles these regions, the same incremental module and ORC execution path used by the OpenMP notebooks can execute them. At that point this notebook should become a real accelerated workload with warmed execution timing—not a frontend-only claim.